# APIM ❤️ Microsoft Foundry

## Microsoft Foundry Hosted Agents (Custom Frameworks) lab
![flow](../../images/ai-foundry-hosted-agents.gif)

This lab deploys a **custom-framework agent** ([Strands](https://strandsagents.com/) or [Pydantic AI](https://ai.pydantic.dev/)) as a **Microsoft Foundry Hosted Agent** and exposes it through Azure API Management (APIM).

The lab is backed by two Microsoft Foundry resources:
- **foundry-models**: hosts the `gpt-5-mini` model deployment, consumed by the agent through APIM.
- **foundry-agents**: hosts the custom-framework runtime as a Hosted Agent (Responses protocol v1.0.0).

### Why run custom frameworks on Foundry Hosted Agents?

- Built-in observability, tracing, and monitoring across agent runtimes.
- Agent Identity and RBAC by default (least-privilege access without embedded secrets).
- Foundry guardrails, governance, evaluations, and red-teaming integration.
- Discovery and reuse across teams through the Foundry control plane.

Clients invoke a hosted agent by specifying the **agent name in the URL path**, so multiple agents are served without changing the APIM configuration:

```
POST {apim-gateway}/hosted-agent-responses/agents/{agentName}/endpoint/protocols/openai/responses?api-version=v1
```

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

Configure resource names, regions, the model deployment, and the hosted-agent framework used throughout this lab. Set `framework` to `'strands'` or `'pydantic'` to choose which custom-framework agent to build and deploy.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = 'swedencentral'

# Two Microsoft Foundry resources:
#   foundry-models: hosts the inference model consumed by the agent
#   foundry-agents: hosts the custom-framework runtime as a Hosted Agent (index 1)
aiservices_config = [
    {"name": "foundry-models", "location": "swedencentral"},
    {"name": "foundry-agents", "location": "swedencentral"},
]

# Model deployed to foundry-models and consumed by the hosted agent through APIM
models_config = [
    {"name": "gpt-5-mini", "publisher": "OpenAI", "version": "2025-08-07",
     "sku": "GlobalStandard", "capacity": 10, "aiservice": "foundry-models"}
]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# API paths
inference_api_path = 'inference'
inference_api_type = 'AzureAI'
hosted_agent_responses_api_path = 'hosted-agent-responses'
foundry_project_name = 'default'
foundry_agent_ai_service_index = 1

frameworks = {
    'strands':  {'agent_name': 'strands-agent',  'image': 'strands-agent'},
    'pydantic': {'agent_name': 'pydantic-agent', 'image': 'pydantic-agent'},
}

build_version = 1

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run('az account show', 'Retrieved az account', 'Failed to get the current az account')

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f'Current user: {current_user}')
    utils.print_info(f'Tenant ID:    {tenant_id}')
    utils.print_info(f'Subscription: {subscription_id}')

# Signed-in user object id is granted the Foundry User (Azure AI User) role on the Foundry resources
output = utils.run('az ad signed-in-user show', 'Retrieved signed-in user', 'Failed to retrieve signed-in user')
foundry_user_object_ids = [output.json_data['id']] if output.success and output.json_data else []

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources deployed in the resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

Deployed resources include:
- Log Analytics Workspace + Application Insights
- Azure API Management (Basicv2) with the inference API and the hosted-agent responses API
- Two Microsoft Foundry resources (`foundry-models` with `gpt-5-mini`, `foundry-agents` for the hosted agent)
- Azure Container Registry for the agent container image
- Role assignments for ACR and Foundry access

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'apimSku':                        {'value': apim_sku},
        'aiServicesConfig':               {'value': aiservices_config},
        'modelsConfig':                   {'value': models_config},
        'apimSubscriptionsConfig':        {'value': apim_subscriptions_config},
        'inferenceAPIPath':               {'value': inference_api_path},
        'inferenceAPIType':               {'value': inference_api_type},
        'foundryProjectName':             {'value': foundry_project_name},
        'foundryAgentAiServiceIndex':     {'value': foundry_agent_ai_service_index},
        'foundryUserObjectIds':           {'value': foundry_user_object_ids},
        'enableHostedAgentResponsesApi':  {'value': True},
        'hostedAgentResponsesApiPath':    {'value': hosted_agent_responses_api_path},
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, the subscription key, the Container Registry name, and the Foundry agent project endpoint used by the remaining steps.

In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} -g {resource_group_name}',
    f'Retrieved deployment: {deployment_name}',
    f'Failed to retrieve deployment: {deployment_name}'
)

if output.success and output.json_data:
    apim_resource_gateway_url      = utils.get_deployment_output(output, 'apimResourceGatewayURL',    'APIM Gateway URL')
    container_registry_name        = utils.get_deployment_output(output, 'containerRegistryName',     'Container Registry Name')
    foundry_agent_project_endpoint = utils.get_deployment_output(output, 'foundryAgentProjectEndpoint', 'Foundry Agent Project Endpoint')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", '"'))
    for subscription in apim_subscriptions:
        utils.print_info(f"Subscription Name: {subscription['name']}")
        utils.print_info(f"Subscription Key:  ****{subscription['key'][-4:]}")
    api_key = apim_subscriptions[0].get('key')

    # Inference endpoint the hosted agent uses to call the model through APIM
    inference_endpoint = f'{apim_resource_gateway_url}/{inference_api_path}/models'

<a id='build'></a>
### 4️⃣ Build and push the agent image to the Container Registry

Build the selected framework's Docker image directly in Azure Container Registry with `az acr build` (no local Docker required). The source is in [`src/frameworks/`](src/frameworks/README.md).

In [ ]:
# Choose your framework: 'strands' or 'pydantic'
# For Hosted agent framework to build and deploy
framework = 'pydantic'

build_version = build_version + 1
agent_name = frameworks[framework]['agent_name']
agent_image_tag = f"{frameworks[framework]['image']}:{build_version}"
framework_src = f"src/frameworks/{framework}"
model_deployment_name = models_config[0]['name']

output = utils.run(
    f'az acr build --registry {container_registry_name} --image {agent_image_tag} {framework_src}',
    f"'{framework}' agent image built and pushed: {agent_image_tag}",
    f"Failed to build and push the '{framework}' agent image"
)

image_uri = f'{container_registry_name}.azurecr.io/{agent_image_tag}'
utils.print_info(f'Image URI: {image_uri}')

<a id='deploy-agent'></a>
### 5️⃣ Create the hosted agent version in Microsoft Foundry

Register the container image as a **Foundry Hosted Agent** (Responses protocol v1.0.0). Foundry auto-assigns a version suffix (`:1`, `:2`, ...). The agent calls the model through the APIM inference API using the injected environment variables.

In [ ]:
output = utils.run(
    'pip install azure-ai-projects==2.3.0 azure-identity -q',
    'Foundry SDK installed',
    'Failed to install the Foundry SDK'
)

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentEndpointProtocol,
    ContainerConfiguration,
)
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

project = AIProjectClient(
    endpoint=foundry_agent_project_endpoint,
    credential=credential,
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=agent_name,
    definition=HostedAgentDefinition(
        protocol_versions=[
            ProtocolVersionRecord(protocol=AgentEndpointProtocol.RESPONSES, version='1.0.0')
        ],
        cpu='1',
        memory='2Gi',
        container_configuration=ContainerConfiguration(image=image_uri),
        environment_variables={
            'AZURE_OPENAI_ENDPOINT': inference_endpoint,
            'AZURE_OPENAI_API_VERSION': '2024-05-01-preview',
            'AZURE_OPENAI_DEPLOYMENT': model_deployment_name,
            'APIM_SUBSCRIPTION_KEY': api_key,
            'LOG_LEVEL': 'INFO',
            #'OTEL_SDK_DISABLED': 'true'
        },
    ),
)

utils.print_ok(f'Agent created: {agent.name}, version: {agent.version}, id: {agent.id}')

<a id='test-direct'></a>
### 🧪 Test the hosted agent directly

Call the Foundry Responses API directly using your Azure CLI credential. This is the baseline test — if it fails, the APIM test will fail too. Wait until the agent reaches the `Running` state before testing.

> **Tip**: Use the [tracing tool](../../tools/tracing.ipynb) to monitor token usage and troubleshoot the [policy](hosted-agent-policy.xml).

In [ ]:
import requests

query = 'Hello! What can you help me with?'
url = f'{foundry_agent_project_endpoint}/agents/{agent_name}/endpoint/protocols/openai/responses'

token = credential.get_token('https://ai.azure.com/.default')
headers = {
    'Authorization': f'Bearer {token.token}',
    'Content-Type': 'application/json',
    'Foundry-Features': 'HostedAgents=V1Preview',
}

utils.print_message(f'Q: {query}')
response = requests.post(url, headers=headers, json={'input': query, 'stream': False}, params={'api-version': 'v1'}, timeout=120)

if response.status_code == 200:
    utils.print_ok('Agent responded successfully (direct)')
    print(json.dumps(response.json(), indent=2))
else:
    utils.print_error(f'Agent returned HTTP {response.status_code}: {response.text}')

<a id='test-apim'></a>
### 🧪 Test the hosted agent through APIM

Route the request through Azure API Management using only the subscription key. APIM injects the managed-identity bearer token, forces `Content-Type: application/json`, and adds the `Foundry-Features` header. The target agent is selected by the `{agentName}` segment in the URL path.

In [ ]:
import requests

query = 'Hello! What can you help me with?'
url = f'{apim_resource_gateway_url}/{hosted_agent_responses_api_path}/agents/{agent_name}/endpoint/protocols/openai/responses'

headers = {
    'api-key': api_key,
    'Content-Type': 'application/json',
    'Foundry-Features': 'HostedAgents=V1Preview',
}

utils.print_message(f'Q: {query}')
response = requests.post(url, headers=headers, json={'input': query, 'stream': False}, params={'api-version': 'v1'}, timeout=120)

if response.status_code == 200:
    utils.print_ok('Agent responded successfully (via APIM)')
    result = response.json()
    agent_text = None
    if result.get('output'):
        content = result['output'][0].get('content', [])
        if content:
            agent_text = content[0].get('text')
    utils.print_ok(f'A: {agent_text}') if agent_text else print(json.dumps(result, indent=2))
else:
    utils.print_error(f'Agent returned HTTP {response.status_code}: {response.text}')

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.